# Erasmus KA1 Mobility Analysis – Data Visualization

## Project Overview

In particular, we will only be looking at the data pertaining university students.
The relevant datasets can be found at the dedicated [European Commission page](https://erasmus-plus.ec.europa.eu/resources-and-tools/statistics-and-factsheets/statistics/for-researchers).

The analysis will be structured across three levels of granularity:
1. **Global** – European-wide trends, how the program has developed in the past decade.
2. **Italy** – A narrower focus on italian universities: number of participants, preferred destinations, distribution across departments and the temporal evolution of the programmes.
3. **UNIMI (UMIL)** – Lastly, a deep-dive into Università degli Studi di Milano and how it has performed compared to other universities in the same city.

This project was developed as part of the Scientific Visualization course taught by Prof. Casiraghi in A.Y. 2025/26.

Valendino Raffaele
Pasquino Riccardo
Opessio Daniele

### Goals of this notebook
This notebook implements the **visualization** of the data needed for the Erasmus KA1 Mobility Analysis project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import plotly.graph_objects as go
import math
import textwrap
import spiderplot as sp
import holoviews as hv
from holoviews import opts
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# 2. Load the Pre‑Processed Dataset

The first step consists in loading the dataset we prepared in the dataProcessing notebook. 
This file contains the cleaned and harmonized records and will serve as the foundation for all subsequent visualizations.

In [ ]:
df = pd.read_csv('Datasets/Processed/Erasmus-Data.csv', index_col=0, low_memory=False)
df = df.reset_index(drop=True)
print(df.shape)
df.head()

___

## 3. Visualizations

### 3.1 European level
Let's first look at how the programme is going on a continental scale. Some relevant information could be wheter the programme has bounced back from the COVID-19 pandemic or what is the outlook in the coming years for the programme.

One thing to note: yearly Erasmus data requires the dataset of the following A.Y. in order to fully account for every participant, as some exchanges might span across more than one academic year, thus making the data for the year 2024 incomplete and forcing us to drop it from our analysis.

In [ ]:
yearly = df
yearly = yearly.groupby('Academic Year').size().reindex(range(2014,2024), fill_value=0)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(yearly.index, yearly.values, marker='o', color='#2a9d8f', linewidth=2, label='Participants')
z = np.polyfit(yearly.index, yearly.values, 1)
ax.plot(yearly.index, np.poly1d(z)(yearly.index), '--', color='gray', label='Linear Trend')
ax.axvspan(2019.9, 2021.1, color='red', alpha=0.25, label='COVID-19')
ax.set_title('Evolution of the Erasmus project')
ax.set_xlabel('Academic Year'); ax.set_ylabel('N. students'); ax.legend()

plt.tight_layout();
plt.savefig("Plots/Evolution of the Erasmus project.png", dpi=800, bbox_inches="tight")
plt.show()

Key insights:
- Erasmus bounced back pretty quickly from the COVID-19 drop.
- The trend is slightly positive.

#### 3.1.1 Choropleth Map of the project

We are interested in the involvement of the various European countries in the project over the past decade. Looking at the students sent allows us to know how involved a specific country was, while looking at the ones received by it allows us to know how attractive it was perceived to be.

Both views are presented through choropleth maps, using a continuous color scale to highlight mobility intensity and make the most active countries immediately recognizable.

In [ ]:
years = sorted(df["Academic Year"].astype(str).str[:4].astype(int).unique())

outgoing_by_year = {}
for year in years:
    df_year = df[df["Academic Year"].astype(str).str[:4].astype(int) == year]
    outgoing = (
        df_year.groupby("Sending Country")
               .size()
               .reset_index(name="outgoing")
               .rename(columns={"Sending Country": "Country"})
    )
    outgoing_by_year[year] = outgoing

world = gpd.read_file("Datasets/ne_110m_admin_0_countries.zip")

world["NAME"] = world["NAME"].replace({
    "Türkiye": "Turkey", 
    "Czechia": "Czech Republic",
    "Bosnia and Herz.": "Bosnia and Herzegovina",
    "Russian Federation": "Russia",
    "Moldova, Republic of": "Moldova"
})

europe_list = [
    "Albania", "Andorra", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria",
    "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Kosovo", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "San Marino", "Serbia",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "Turkey", "Ukraine", "United Kingdom",
    "Vatican"
]
europe = world[world["NAME"].isin(europe_list)].copy()

GLOBAL_MAX = max(d["outgoing"].max() for d in outgoing_by_year.values() if not d.empty)
GLOBAL_MAX = math.ceil(GLOBAL_MAX / 1000) * 1000
COLORSCALE = "YlOrRd"  
COUNTRY_LINE_COLOR = "white"

def build_year_data(year):
    year_df = outgoing_by_year[year].copy()
    
    merged = europe.merge(year_df, left_on="NAME", right_on="Country", how="left")
    merged = merged.sort_values("NAME").reset_index(drop=True)
    
    merged["hover"] = (
        "<b>" + merged["NAME"] + "</b><br>" +
        "Outgoing students: " + 
        merged["outgoing"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "No data")
    )
    return merged

frames_data = {year: build_year_data(year) for year in years}

def make_trace(data_year, showscale=True):
    return go.Choropleth(
        locations=data_year["NAME"],
        z=data_year["outgoing"],
        text=data_year["hover"],
        hovertemplate="%{text}<extra></extra>",
        locationmode="country names",
        colorscale=COLORSCALE,
        zmin=0,
        zmax=GLOBAL_MAX,
        marker_line_color=COUNTRY_LINE_COLOR,
        marker_line_width=0.8,
        showscale=showscale,
        colorbar=dict(
            title=dict(text="Students", font=dict(size=12, color="#444")),
            thickness=15,
            len=0.85,
            x=0.92,
            y=0.5,
            yanchor="middle",
            tickfont=dict(size=11, color="#555"),
            outlinewidth=0,
        ),
    )

frames = []
for year in years:
    frames.append(
        go.Frame(
            data=[make_trace(frames_data[year])],
            name=str(year),
            layout=go.Layout(
                title=dict(
                    text=f"<b>Outgoing Erasmus Students per Country A.Y. {year}</b>"
                )
            ),
        )
    )

first_year = years[0]
fig = go.Figure(
    data=[make_trace(frames_data[first_year])],
    frames=frames,
)


fig.update_layout(
    title=dict(
        text=f"<b>Outgoing Erasmus Students per Country A.Y. {first_year}</b>",
        x=0.5, xanchor="center", y=0.94,
        font=dict(size=22, family="Arial", color="#2b2b2b"),
    ),
    font=dict(family="Helvetica Neue, Arial", color="#2b2b2b"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    width=1000,
    height=750,
    margin=dict(l=30, r=30, t=110, b=20),

    geo=dict(
        showframe=False,
        showcoastlines=False,
        showcountries=True,
        countrycolor="white",
        showland=True,
        landcolor="#eef0f2",   
        showocean=False,
        projection_type="natural earth",
        center=dict(lat=53, lon=15),
        lataxis_range=[34, 71],   
        lonaxis_range=[-22, 45],  
        bgcolor="white",
    ),

    sliders=[{
        "active": 0,
        "currentvalue": {
            "prefix": "Academic Year: ",
            "font": {"size": 13, "family": "Arial"},
        },
        "pad": {"t": 30, "b": 15},
        "x": 0.12, "len": 0.76,
        "transition": {"duration": 400, "easing": "cubic-in-out"},
        "steps": [
            {
                "method": "animate",
                "args": [[str(year)], {
                    "frame": {"duration": 400, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 400},
                }],
                "label": str(year),
            }
            for year in years
        ],
    }],

    updatemenus=[{
        "type": "buttons",
        "direction": "left",
        "x": 0.12, "y": 0.02,
        "xanchor": "left", "yanchor": "bottom",
        "showactive": False,
        "font": {"size": 12, "family": "Arial"},
        "bgcolor": "white",
        "bordercolor": "#ddd",
        "borderwidth": 1,
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 800, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 400},
                }],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                }],
            },
        ],
    }],
)

fig.show()

In [ ]:
frame_output_dir = Path("Plots/outgoing_frames")
frame_output_dir.mkdir(parents=True, exist_ok=True)

for year in years:
    frame_fig = go.Figure(data=[make_trace(frames_data[year], showscale=True)])
    frame_fig.update_layout(
        title=dict(
            text=f"<b>Outgoing Erasmus Students per Country A.Y. {year}</b>",
            x=0.5,
            xanchor="center",
            y=0.94,
            font=dict(size=22, family="Arial", color="#2b2b2b"),
        ),
        font=dict(family="Helvetica Neue, Arial", color="#2b2b2b"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1000,
        height=750,
        margin=dict(l=30, r=30, t=110, b=20),
        geo=dict(
            showframe=False,
            showcoastlines=False,
            showcountries=True,
            countrycolor="white",
            showland=True,
            landcolor="#eef0f2",
            showocean=False,
            projection_type="natural earth",
            center=dict(lat=53, lon=15),
            lataxis_range=[34, 71],
            lonaxis_range=[-22, 45],
            bgcolor="white",
        ),
    )
    frame_fig.write_image(frame_output_dir / f"Outgoing Erasmus Students per Country A.Y. {year}.png", scale=2)

In [ ]:
years = sorted(df["Academic Year"].astype(str).str[:4].astype(int).unique())

incoming_by_year = {}
for year in years:
    df_year = df[df["Academic Year"].astype(str).str[:4].astype(int) == year]
    incoming = (
        df_year.groupby("Receiving Country")
               .size()
               .reset_index(name="incoming")
               .rename(columns={"Receiving Country": "Country"})
    )
    incoming_by_year[year] = incoming


# Load and clean the map geometry
world = gpd.read_file("Datasets/ne_110m_admin_0_countries.zip")

# Standardize country names
world["NAME"] = world["NAME"].replace({
    "Türkiye": "Turkey", 
    "Czechia": "Czech Republic",
    "Bosnia and Herz.": "Bosnia and Herzegovina",
    "Russian Federation": "Russia",
    "Moldova, Republic of": "Moldova"
})

europe_list = [
    "Albania", "Andorra", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria",
    "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Kosovo", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "San Marino", "Serbia",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "Turkey", "Ukraine", "United Kingdom",
    "Vatican"
]

europe = world[world["NAME"].isin(europe_list)].copy()

GLOBAL_MAX = max(d["incoming"].max() for d in incoming_by_year.values() if not d.empty)
GLOBAL_MAX = math.ceil(GLOBAL_MAX / 1000) * 1000
COLORSCALE = "YlOrRd"  
COUNTRY_LINE_COLOR = "white"

def build_year_data(year):
    year_df = incoming_by_year[year].copy()
    
    merged = europe.merge(year_df, left_on="NAME", right_on="Country", how="left")
    merged = merged.sort_values("NAME").reset_index(drop=True)
    
    merged["hover"] = (
        "<b>" + merged["NAME"] + "</b><br>" +
        "Incoming students: " + 
        merged["incoming"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "No data")
    )
    return merged

frames_data = {year: build_year_data(year) for year in years}

def make_trace(data_year, showscale=True):
    return go.Choropleth(
        locations=data_year["NAME"],
        z=data_year["incoming"],
        text=data_year["hover"],
        hovertemplate="%{text}<extra></extra>",
        locationmode="country names",
        colorscale=COLORSCALE,
        zmin=0,
        zmax=GLOBAL_MAX,
        marker_line_color=COUNTRY_LINE_COLOR,
        marker_line_width=0.8,
        showscale=showscale,
        colorbar=dict(
            title=dict(text="Students", font=dict(size=12, color="#444")),
            thickness=15,
            len=0.9,               # Proportionally scales colorbar height relative to the graph
            x=0.92,                # Adjusted inwards so it stays safe inside the width bounds
            y=0.5,                 # Centers vertically
            yanchor="middle",
            tickfont=dict(size=11, color="#555"),
            outlinewidth=0,
        ),
    )


# Frames
frames = []
for year in years:
    frames.append(
        go.Frame(
            data=[make_trace(frames_data[year])],
            name=str(year),
            layout=go.Layout(
                # Update the layout title dynamically for each year frame
                title=dict(
                    text=f"<b>Incoming Erasmus Students per Country A.Y {year}</b>"
                )
            ),
        )
    )

first_year = years[0]
fig = go.Figure(
    data=[make_trace(frames_data[first_year])],
    frames=frames,
)

# Global Layout Configuration
fig.update_layout(
    title=dict(
        text=f"<b>Incoming Erasmus Students per Country A.Y {first_year}</b>",
        x=0.5, xanchor="center", y=0.94,
        font=dict(size=22, family="Arial", color="#2b2b2b"),
    ),
    font=dict(family="Helvetica Neue, Arial", color="#2b2b2b"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    width=1000,
    height=750,
    margin=dict(l=30, r=30, t=110, b=20),

    geo=dict(
        showframe=False,
        showcoastlines=False,
        showcountries=True,
        countrycolor="white",
        showland=True,
        landcolor="#eef0f2",   
        showocean=False,
        projection_type="natural earth",
        center=dict(lat=53, lon=15),
        lataxis_range=[34, 71],   
        lonaxis_range=[-22, 45],  
        bgcolor="white",
    ),

    sliders=[{
        "active": 0,
        "currentvalue": {
            "prefix": "Academic Year: ",
            "font": {"size": 13, "family": "Arial"},
        },
        "pad": {"t": 30, "b": 15},
        "x": 0.12, "len": 0.76,
        "transition": {"duration": 400, "easing": "cubic-in-out"},
        "steps": [
            {
                "method": "animate",
                "args": [[str(year)], {
                    "frame": {"duration": 400, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 400},
                }],
                "label": str(year),
            }
            for year in years
        ],
    }],

    updatemenus=[{
        "type": "buttons",
        "direction": "left",
        "x": 0.12, "y": 0.02,
        "xanchor": "left", "yanchor": "bottom",
        "showactive": False,
        "font": {"size": 12, "family": "Arial"},
        "bgcolor": "white",
        "bordercolor": "#ddd",
        "borderwidth": 1,
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 800, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 400},
                }],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                }],
            },
        ],
    }],
)

fig.show()

In [ ]:
frame_output_dir = Path("Plots/incoming_frames")
frame_output_dir.mkdir(parents=True, exist_ok=True)

for year in years:
    frame_fig = go.Figure(data=[make_trace(frames_data[year], showscale=True)])
    frame_fig.update_layout(
        title=dict(
            text=f"<b>Incoming Erasmus Students per Country A.Y. {year}</b>",
            x=0.5,
            xanchor="center",
            y=0.94,
            font=dict(size=22, family="Arial", color="#2b2b2b"),
        ),
        font=dict(family="Helvetica Neue, Arial", color="#2b2b2b"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1000,
        height=750,
        margin=dict(l=30, r=30, t=110, b=20),
        geo=dict(
            showframe=False,
            showcoastlines=False,
            showcountries=True,
            countrycolor="white",
            showland=True,
            landcolor="#eef0f2",
            showocean=False,
            projection_type="natural earth",
            center=dict(lat=53, lon=15),
            lataxis_range=[34, 71],
            lonaxis_range=[-22, 45],
            bgcolor="white",
        ),
    )
    frame_fig.write_image(frame_output_dir / f"Incoming Erasmus Students per Country A.Y. {year}.png", scale=2)

___

#### 3.1.2 Heatmap of the Universities

Now that we know how each country fared in the past decade of the programme, we are interested in knowing which institutions were the most involved, as well as knowing which ones were considered to be more attractive by the students.

In [ ]:
top_countries = df['Sending Country'].value_counts().head(10).index

univ_column = 'Sending Organization'
is_valid_org = (
    df[univ_column].notna() &                                      
    (df[univ_column].astype(str).str.strip() != '') &              
    (df[univ_column].astype(str).str.strip() != '-') &             
    (~df[univ_column].astype(str).str.upper().isin(['N/D', 'N/A', 'UNKNOWN'])) 
)
df_valid_orgs = df[is_valid_org]

# Build Matrices
mat = pd.DataFrame(index=top_countries, columns=['#1', '#2', '#3'], dtype=float)
labels = pd.DataFrame(index=top_countries, columns=['#1', '#2', '#3'], dtype=object)

for c in top_countries:
    sub = df_valid_orgs[df_valid_orgs['Sending Country'] == c]
    top3 = sub[univ_column].value_counts().head(3)
    
    for i, (org, n) in enumerate(top3.items()):
        mat.loc[c, f'#{i+1}'] = n
        
        wrapped_org_name = textwrap.fill(str(org), width = 25, max_lines = 4)
        
        labels.loc[c, f'#{i+1}'] = f"{wrapped_org_name}\n(n = {int(n):,})"

mat = mat.fillna(0)
labels = labels.fillna('')

# Plot
plt.figure(figsize=(15, 12)) 
plt.rcParams['font.family'] = 'Arial'

ax = sns.heatmap(
    mat, 
    annot=labels, 
    fmt='', 
    cmap='YlGnBu', 
    linewidths=0.8, 
    linecolor='white',
    cbar_kws={'label': 'N. Students'}, 
    annot_kws={'fontsize': 10, 'weight': 'normal'}
    )

ax.set_title('Top 3 most involved Universities per Country', 
             fontsize=14, pad=15, weight='bold', color='#2b2b2b')
ax.set_ylabel('')
ax.set_xlabel('')
ax.tick_params(axis='both', which='major', labelsize=11, labelcolor='#2b2b2b')

plt.tight_layout()
plt.savefig("Plots/Top 3 most involved Universities per Country.png", dpi=800, bbox_inches="tight")
plt.show()

Key insights:
- The density of universities reflects in the number of students per university that go on an exchange. France, despite being the 1st in terms of participants students, shows average number of participants per university, while other countries such as Portugal, show a really high number for just 1 institution.

In [ ]:
top_countries = df['Receiving Country'].value_counts().head(10).index

univ_column = 'Receiving Organization'
is_valid_org = (
    df[univ_column].notna() &                                      
    (df[univ_column].astype(str).str.strip() != '') &              
    (df[univ_column].astype(str).str.strip() != '-') &             
    (~df[univ_column].astype(str).str.upper().isin(['N/D', 'N/A', 'UNKNOWN'])) 
)
df_valid_orgs = df[is_valid_org]

mat = pd.DataFrame(index=top_countries, columns=['#1', '#2', '#3'], dtype=float)
labels = pd.DataFrame(index=top_countries, columns=['#1', '#2', '#3'], dtype=object)

for c in top_countries:
    sub = df_valid_orgs[df_valid_orgs['Receiving Country'] == c]
    top3 = sub[univ_column].value_counts().head(3)
    
    for i, (org, n) in enumerate(top3.items()):
        mat.loc[c, f'#{i+1}'] = n
        
        wrapped_org_name = textwrap.fill(str(org), width = 25, max_lines = 4)
        
        labels.loc[c, f'#{i+1}'] = f"{wrapped_org_name}\n(n = {int(n):,})"

mat = mat.fillna(0)
labels = labels.fillna('')

# Plot
plt.figure(figsize=(15, 12)) 
plt.rcParams['font.family'] = 'Arial'

ax = sns.heatmap(
    mat, 
    annot=labels, 
    fmt='', 
    cmap='YlGnBu', 
    linewidths=0.8, 
    linecolor='white',
    cbar_kws={'label': 'N. Students'}, 
    annot_kws={'fontsize': 10, 'weight': 'normal'}
    )

ax.set_title('Top 3 most attractive Universities per Country', 
             fontsize=14, pad=15, weight='bold', color='#2b2b2b')
ax.set_ylabel('')
ax.set_xlabel('')
ax.tick_params(axis='both', which='major', labelsize=11, labelcolor='#2b2b2b')

plt.tight_layout()
plt.savefig("Plots/Top 3 most attractive Universities per Country.png", dpi=800, bbox_inches="tight")
plt.show()

___

#### 3.1.3 Chord Diagram for the exchanges

To study how each nation interacts within the Erasmus network, we use a chord diagram built with the Holoviews layout. 
The diagram visualises the exchanges among the ten most involved countries, showing each state on a circular axis and the corresponding flows as curved ribbons.


In [ ]:
hv.extension('bokeh')

top10 = df['Sending Country'].value_counts().head(10).index
flows_ct = df.groupby(['Sending Country', 'Receiving Country']).size().rename('n').reset_index()
flows_ct = flows_ct[flows_ct['Sending Country'].isin(top10) & flows_ct['Receiving Country'].isin(top10)]
flows_ct = flows_ct[flows_ct['n'] >= 2].sort_values('n', ascending=False)

chord_data = flows_ct.rename(columns={
    'Sending Country': 'source',
    'Receiving Country': 'target',
    'n': 'value'
})

palette = ['#e07a5f', '#3d5a80', '#81b29a', '#f2cc8f', '#e9c46a', 
           '#457b9d', '#2a9d8f', '#e63946', '#f4a261', '#264653']


chord = hv.Chord(chord_data)

chord.opts(
    opts.Chord(
        title="Erasmus exchanges among the 10 most active countries",
        cmap=palette,
        edge_cmap=palette,
        edge_color=hv.dim('source').str(),
        node_color=hv.dim('index').str(),
        edge_alpha=0.6,
        edge_hover_line_color='black',
        node_size=15,
        labels='index',
        width=700,
        height=700,
    )
)
# Render
chord

Key insight:
- Southern Europe prefers to exchange within itself.

In [ ]:
hv.extension('matplotlib')

chord_mpl = hv.Chord(chord_data).opts(
    opts.Chord(
        title="Erasmus exchanges among the 10 most active countries",
        cmap=palette,
        edge_cmap=palette,
        edge_color=hv.dim('source').str(),
        node_color=hv.dim('index').str(),
        labels='index',
        fig_size=300,
    )
)

hv.save(chord_mpl, 'Plots/Erasmus exchanges among the 10 most active countries.png', fmt='png', dpi=800)

___

### 3.2 Italian level

Looking at the chords, the ones related to Italy actually stroke a chord (pun intended) in us: Italy sure is quite active in the programme.

We are thus interested in looking at how the programme has evolved in our country.

In [ ]:
it = df[df['Sending Country']=='Italy']
yearly_it = it.groupby('Academic Year').size().reindex(range(2014,2024), fill_value=0)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(yearly_it.index, yearly_it.values, marker='o', color='#2a9d8f', linewidth=2, label='Students')
z = np.polyfit(yearly_it.index, yearly_it.values, 1)
ax.plot(yearly_it.index, np.poly1d(z)(yearly_it.index), '--', color='gray', label='Linear Trend')
ax.axvspan(2019.9, 2021.1, color='red', alpha=0.25, label='COVID-19')
ax.set_title('Evolution of the participation of italian students to Erasmus exchanges')
ax.set_xlabel('Academic Year'); ax.set_ylabel('N. students'); ax.legend()

plt.tight_layout();
plt.savefig("Plots/Evolution of the participation of italian students to Erasmus exchanges.png", dpi=800, bbox_inches="tight")
plt.show()

In [ ]:
it = df[df['Receiving Country']=='Italy']
yearly_it = it.groupby('Academic Year').size().reindex(range(2014,2024), fill_value=0)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(yearly_it.index, yearly_it.values, marker='o', color='#2a9d8f', linewidth=2, label='Students')
z = np.polyfit(yearly_it.index, yearly_it.values, 1)
ax.plot(yearly_it.index, np.poly1d(z)(yearly_it.index), '--', color='gray', label='Linear Trend')
ax.axvspan(2019.9, 2021.1, color='red', alpha=0.25, label='COVID-19')
ax.set_title('Evolution of the attractivity of Italy for Erasmus exchanges')
ax.set_xlabel('Academic Year'); ax.set_ylabel('N. students'); ax.legend()

plt.tight_layout();
plt.savefig("Plots/Evolution of the attractivity of Italy for Erasmus exchanges.png", dpi=800, bbox_inches="tight")
plt.show()

Key insights:
- The number of incoming students is lower than that of outgoing students.
- Attractivity is increasing faster compared to the participation of local students to the programme.

___

#### 3.2.1 Spiderplot of the biggest cities

To obtain a more detailed view of the destination of the incoming mobility, we looked at the performance of the five main city campuses: Turin, Milan, Bologna, Rome and Naples. 

For each academic year, incoming records associated with the universities located in these cities were aggregated to produce yearly counts and said counts were then plotted into a Spider Plot.

One thing to note: the various universities located in Rome changed the way they submitted their results in 2022, leading to the loss of data for University of La Sapienza in the years past 2022. It can be clearly seen in the plot, as Rome drops down to levels closer to Naples while it competed with Milan and Bologna before 2022.

In [ ]:
cities = {
    'Turin': ["UNIVERSITA DEGLI STUDI DI TORINO", "POLITECNICO DI TORINO"],
    'Milan': ["POLITECNICO DI MILANO", "UNIVERSITA DEGLI STUDI DI MILANO", "UNIVERSITA COMMERCIALE LUIGI BOCCONI"],
    'Bologna': ["ALMA MATER STUDIORUM - UNIVERSITA DI BOLOGNA"],
    'Rome': ["UNIVERSITA DEGLI STUDI DI ROMA LA SAPIENZA", "UNIVERSITA DEGLI STUDI ROMA TRE", "UNIVERSITA DEGLI STUDI DI ROMA TOR VERGATA"],
    'Naples': ["UNIVERSITA DEGLI STUDI DI NAPOLI FEDERICO II.", "UNIVERSITA DEGLI STUDI DI NAPOLI PARTHENOPE", "UNIVERSITA DEGLI STUDI DI NAPOLI L'ORIENTALE"]
}

years = list(range(2014, 2024))

rows = []
for year in years:
    for city, universities in cities.items():
        value = 0
        for uni in universities:
            df_uni = pd.DataFrame()
            df_uni = df[(df['Receiving Organization'] == uni) & (df['Academic Year'] == year)].copy()
            value += len(df_uni)
        rows.append({'Anno': str(year), 'Citta': city, 'Valore': value})

data = pd.DataFrame(rows)

palette = {'Turin': '#1d3557', 'Milan': '#e63946', 'Bologna': '#2a9d8f',
           'Rome': '#e9c46a', 'Naples': '#6a4c93'}

plt.figure(figsize=(11, 11))
plot = sp.spiderplot(x="Anno", y="Valore", hue="Citta", legend=True,
                      data=data, palette=palette, rref=0, alpha=1)
plot.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10, prop={'size': 13})
plot.set_title("Incoming Erasmus per city campuses",
               y=1.08, fontsize=20, horizontalalignment='center', verticalalignment='center')
plt.tight_layout()
plt.savefig("Plots/Incoming Erasmus per city campuses.png", dpi=800, bbox_inches="tight")
plt.show()

___

### 3.3 University of Milan level

Looking at how Milan battled with Bologna to get the number one spot as the most desired destination for an Erasmus Exchange in our country, we thought it would be interesting to look at how our university is performing compared to the other ones in the city.

In [ ]:
unimi_exact = ["UNIVERSITA DEGLI STUDI DI MILANO"]
polimi_exact = ["POLITECNICO DI MILANO"]
bocconi_exact = ["UNIVERSITA COMMERCIALE LUIGI BOCCONI"]
bicocca_exact = ["UNIVERSITA' DEGLI STUDI DI MILANO-BICOCCA"]
cattolica_exact = ["UNIVERSITA CATTOLICA DEL SACRO CUORE"]
iulm_exact = ["LIBERA UNIVERSITA DI LINGUE E COMUNICAZIONE IULM", "IULM"]
raffaele_exact = ["UNIVERSITA VITA-SALUTE SAN RAFFAELE"]

all_exact_names = (
    unimi_exact + polimi_exact + bocconi_exact + 
    bicocca_exact + cattolica_exact + iulm_exact + raffaele_exact
)

milan_senders = df[df['Sending Organization'].isin(all_exact_names)]
outgoing_counts = milan_senders['Sending Organization'].value_counts()

milan_receivers = df[df['Receiving Organization'].isin(all_exact_names)]
incoming_counts = milan_receivers['Receiving Organization'].value_counts()


df_out = outgoing_counts.reset_index(name='Outgoing')
df_out.columns = ['University', 'Outgoing']

df_in = incoming_counts.reset_index(name='Incoming')
df_in.columns = ['University', 'Incoming']

df_milan = pd.merge(df_out, df_in, on='University', how='outer').fillna(0)

def clean_milan_names(name):
    if name in polimi_exact:
        return 'PoliMi'
    elif name in bocconi_exact:
        return 'Bocconi'
    elif name in bicocca_exact:
        return 'UniMiB'
    elif name in cattolica_exact:
        return 'Cattolica'
    elif name in unimi_exact:
        return 'UNIMI'
    elif name in iulm_exact:
        return 'IULM'
    elif name in raffaele_exact:
        return 'San Raffaele'
    else:
        return textwrap.fill(str(name), width=25)

df_milan['Cleaned University'] = df_milan['University'].apply(clean_milan_names)

df_milan_grouped = df_milan.groupby('Cleaned University')[['Outgoing', 'Incoming']].sum().reset_index()


df_milan_grouped['Total'] = df_milan_grouped['Outgoing'] + df_milan_grouped['Incoming']
df_milan_top = df_milan_grouped.sort_values(by='Total', ascending=False).head(7)


df_molten = df_milan_top.melt(
    id_vars='Cleaned University', 
    value_vars=['Outgoing', 'Incoming'], 
    var_name='Tipologia Studenti', 
    value_name='Conteggio'
)

plt.figure(figsize=(12, 7.5))
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')

ax = sns.barplot(
    data=df_molten,
    x='Cleaned University',
    y='Conteggio',
    hue='Tipologia Studenti',
    palette=['#3d5a80', '#9b5de5']
)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height):,}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=9.5, color='#2b2b2b', weight='semibold',
                    xytext=(0, 4),
                    textcoords='offset points')

ax.set_title('Top 7 Universities of Milan with respect to Erasmus Exchanges', 
             fontsize=14, pad=20, weight='bold', color='#2b2b2b')
ax.set_xlabel('University', fontsize=12, labelpad=12, weight='semibold', color='#2b2b2b')
ax.set_ylabel('N. students', fontsize=12, labelpad=12, weight='semibold', color='#2b2b2b')

plt.xticks(rotation=0, horizontalalignment='center', fontsize=10, color='#2b2b2b')
plt.yticks(fontsize=10, color='#2b2b2b')
plt.legend(title='Direction', title_fontsize='11', fontsize='10', frameon=True)

plt.tight_layout()
plt.savefig("Plots/Top 7 Universities of Milan with respect to Erasmus Exchanges.png", dpi=800, bbox_inches="tight")
plt.show()

Key insights:
- UNIMI has a low Outgoing / Incoming ratio. This suggest possible attractivity issues. Let's take a closer look

#### 3.3.1 UniMi

In [ ]:
it = df[df['Sending Organization']=='UNIVERSITA DEGLI STUDI DI MILANO']
yearly_it = it.groupby('Academic Year').size().reindex(range(2014,2024), fill_value=0)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(yearly_it.index, yearly_it.values, marker='o', color='#2a9d8f', linewidth=2, label='Students')
z = np.polyfit(yearly_it.index, yearly_it.values, 1)
ax.plot(yearly_it.index, np.poly1d(z)(yearly_it.index), '--', color='gray', label='Linear Trend')
ax.axvspan(2019.9, 2021.1, color='red', alpha=0.25, label='COVID-19')
ax.set_title('Evolution of the participation to Erasmus exchanges at UniMi')
ax.set_xlabel('Academic Year'); ax.set_ylabel('N. students'); ax.legend()

plt.tight_layout(); 
plt.savefig("Plots/Evolution of the participation to Erasmus exchanges at UniMi.png", dpi=800, bbox_inches="tight")
plt.show()

In [ ]:
it = df[df['Receiving Organization']=='UNIVERSITA DEGLI STUDI DI MILANO']
yearly_it = it.groupby('Academic Year').size().reindex(range(2014,2024), fill_value=0)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(yearly_it.index, yearly_it.values, marker='o', color='#2a9d8f', linewidth=2, label='Students')
z = np.polyfit(yearly_it.index, yearly_it.values, 1)
ax.plot(yearly_it.index, np.poly1d(z)(yearly_it.index), '--', color='gray', label='Linear Trend')
ax.axvspan(2019.9, 2021.1, color='red', alpha=0.25, label='COVID-19')
ax.set_title('Evolution of the attractivity of UniMi for Erasmus exchanges')
ax.set_xlabel('Academic Year'); ax.set_ylabel('N. students'); ax.legend()

plt.tight_layout();
plt.savefig("Plots/Evolution of the attractivity of UniMi for Erasmus exchanges.png", dpi=800, bbox_inches="tight")
plt.show()

It might be interesting to look at what the incoming students are studying:

In [ ]:
unimi_incoming = df[df['Receiving Organization'] == "UNIVERSITA DEGLI STUDI DI MILANO"].copy()

# Count per Field of Education (simil CdL)
field_counts = unimi_incoming['Field of Education'].value_counts().reset_index()
field_counts.columns = ['Field of Education', 'Count']

field_counts_top = field_counts.head(15).copy()

field_counts_top['Wrapped Field'] = field_counts_top['Field of Education'].apply(
    lambda x: textwrap.fill(str(x), width=28)
)

# Plot
plt.figure(figsize=(12, 6.5))
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')

ax = sns.barplot(
    data=field_counts_top,
    y='Wrapped Field',
    x='Count',
    color='#3d5a80',
    edgecolor='none'
)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f' {int(width):,}',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    fontsize=10, color='#2b2b2b', weight='semibold',
                    xytext=(4, 0),
                    textcoords='offset points')

# Typography and aesthetics adjustments
ax.set_title('Incoming students at UniMi divided by area of study', 
             fontsize=14, pad=20, weight='bold', color='#2b2b2b')
ax.set_xlabel('N. students', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')
ax.set_ylabel('Area of study', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')


ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.yaxis.grid(False)

plt.tight_layout()
plt.savefig("Plots/Incoming students at UniMi divided by area of study.png", dpi=800, bbox_inches="tight")
plt.show()

___

In [ ]:
polimi_incoming = df[df['Receiving Organization'] == "POLITECNICO DI MILANO"].copy()


field_counts = polimi_incoming['Field of Education'].value_counts().reset_index()
field_counts.columns = ['Field of Education', 'Count']

field_counts_top = field_counts.head(15).copy()

field_counts_top['Wrapped Field'] = field_counts_top['Field of Education'].apply(
    lambda x: textwrap.fill(str(x), width=28)
)

plt.figure(figsize=(12, 6.5))
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')


ax = sns.barplot(
    data=field_counts_top,
    y='Wrapped Field',
    x='Count',
    color='#3d5a80',
    edgecolor='none'
)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f' {int(width):,}',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    fontsize=10, color='#2b2b2b', weight='semibold',
                    xytext=(4, 0),
                    textcoords='offset points')

# Typography and aesthetics adjustments
ax.set_title('Incoming students at PoliMi divided by area of study', 
             fontsize=14, pad=20, weight='bold', color='#2b2b2b')
ax.set_xlabel('N. students', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')
ax.set_ylabel('Area of study', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')

# Remove vertical clutter
ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.yaxis.grid(False)

plt.tight_layout()
plt.savefig("Plots/Incoming students at PoliMi divided by area of study.png", dpi=800, bbox_inches="tight")
plt.show()

In [ ]:
UniBo_incoming = df[df['Receiving Organization'] == "ALMA MATER STUDIORUM - UNIVERSITA DI BOLOGNA"].copy()


field_counts = UniBo_incoming['Field of Education'].value_counts().reset_index()
field_counts.columns = ['Field of Education', 'Count']

field_counts_top = field_counts.head(15).copy()

field_counts_top['Wrapped Field'] = field_counts_top['Field of Education'].apply(
    lambda x: textwrap.fill(str(x), width=28)
)

plt.figure(figsize=(12, 6.5))
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')


ax = sns.barplot(
    data=field_counts_top,
    y='Wrapped Field',
    x='Count',
    color='#3d5a80',
    edgecolor='none'
)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f' {int(width):,}',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    fontsize=10, color='#2b2b2b', weight='semibold',
                    xytext=(4, 0),
                    textcoords='offset points')

# Typography and aesthetics adjustments
ax.set_title('Incoming students at UniBo divided by area of study', 
             fontsize=14, pad=20, weight='bold', color='#2b2b2b')
ax.set_xlabel('N. students', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')
ax.set_ylabel('Area of study', fontsize=11, labelpad=10, weight='semibold', color='#2b2b2b')

# Remove vertical clutter
ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.yaxis.grid(False)

plt.tight_layout()
plt.savefig("Plots/Incoming students at UniBo divided by area of study.png", dpi=800, bbox_inches="tight")
plt.show()